## Structured Output and Pydantic



In [3]:
# executing normal model
import os
from langchain.chat_models import init_chat_model
from IPython.display import display,Markdown
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:openai/gpt-oss-120b")

pydantic validation

In [26]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title : str=Field(description="Title of the movie")
    year : int=Field(description="Moive release year")
    hero : str=Field(description="Hero of the movie")
    director: str=Field(description="director of the movie")

In [27]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002DC46B37110>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002DC46B37B10>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'Title

In [38]:
# it will show raw messages also formated messages
model_with_structure_raw=model.with_structured_output(Movie,include_raw=True)
res=model_with_structure_raw.invoke("provide details about kgf movie")
res


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks "provide details about kgf movie". Likely they want details like director, hero, title, year. We have a function Movie that expects director, hero, title, year. So we should call that function with appropriate data for KGF. KGF (Kolar Gold Fields) is a Kannada film. The first part released 2018, director Prashanth Neel, hero Yash, title "KGF: Chapter 1", year 2018. Maybe also mention Chapter 2 (2022). But function expects single movie. Could we provide both? Probably call function twice? Might just provide details manually without function. The instructions: we have function to provide movie details. Probably we should call function to get structured data. Let\'s call function with KGF: Chapter 1.', 'tool_calls': [{'id': 'fc_d1732d41-fb21-4377-b63f-9442fe14ef76', 'function': {'arguments': '{"director":"Prashanth Neel","hero":"Yash","title":"KGF: Chapter 1","year":2018}', 'name': 'Movie'}, 'type': 'function

In [28]:
res=model.invoke("say something about kgf movie in 2 line")
display(Markdown(res.content))

KGF is a high‑octane Indian action saga that chronicles Rocky’s rise from the streets of Mumbai to become the feared overlord of the Kolar Gold Mine. Its gritty visuals, pulsating soundtrack, and larger‑than‑life heroics have turned it into a cult phenomenon across the subcontinent.

In [34]:
res=model_with_structure.invoke("provide details about kgf movie")
res


Movie(title='KGF: Chapter 1', year=2018, hero='Yash', director='Prashanth Neel')

## nested Structure


In [41]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str=Field(description="name of the Hero")
    role: str=Field(description="role played in the movie")
    
class Movie(BaseModel):
    title : str=Field(description="Title of the movie")
    year : int=Field(description="Moive release year")
    cast: list[Actor]
    director: str=Field(description="director of the movie")


In [43]:
model_with_structure=model.with_structured_output(Movie)
res=model_with_structure.invoke("provide details about kgf movie")
res

Movie(title='KGF: Chapter 1', year=2018, cast=[Actor(name='Yash', role='Rocky'), Actor(name='Sanjay Dutt', role='Adheera'), Actor(name='Raveena Tandon', role='Ramika Sen'), Actor(name='Ramachandra Raju', role='Garuda'), Actor(name='Anant Nag', role='Mohan Lal')], director='Prashanth Neel')

## TypeDict

#### simple alternative for pydantic and ideal when don't need runtime validation

In [44]:
from typing_extensions import Annotated, TypedDict

class MovieDict(TypedDict):
    """ A movie with details"""
    title : Annotated[str,...,"Title of the movie"]
    year : Annotated[int,...,"Moive release year"]
    hero : Annotated[str,...,"Hero of the movie"]
    director: Annotated[str,...,"director of the movie"]

In [46]:
model_with_structure=model.with_structured_output(MovieDict)
res=model_with_structure.invoke("provide details about kgf movie")
res

{'director': 'Prashanth Neel',
 'hero': 'Yash',
 'title': 'KGF: Chapter 1',
 'year': 2018}

In [52]:
from typing_extensions import Annotated, TypedDict

class Actor(TypedDict):
    name: str
    role: str
    
class MovieDetails(TypedDict):
    title : str
    year : int
    cast: list[Actor]
    director: str


In [53]:
model_with_structure=model.with_structured_output(MovieDetails)
res=model_with_structure.invoke("provide details about kgf movie")
res

{'cast': [{'name': 'Yash', 'role': 'Rocky'},
  {'name': 'Srinidhi Shetty', 'role': 'Reena'},
  {'name': 'Ramachandra Raju', 'role': 'Garuda'},
  {'name': 'Anant Nag', 'role': 'Anand Ingal'},
  {'name': 'Achyuth Kumar', 'role': 'Kalyan'}],
 'director': 'Prashanth Neel',
 'title': 'KGF: Chapter 1',
 'year': 2018}

## model profile

In [54]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

## Data classes

In [2]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """ contact information for a person"""
    name : str
    email: str
    phone: str

agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

result=agent.invoke({
    "messages":[{"role":"user", "content": "Extract contact info from John, john@example.com, 1234567899"}]
})
result["structured_response"]


ContactInfo(name='John', email='john@example.com', phone='1234567899')